# Gen AI Eval - Multi-turn Agent Eval, User Simulation, Metric Registration, Auto-Loss Analysis

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fevaluation%2Fmulti_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/multi_turn_agent_evaluation_with_user_simulation_metric_registration_auto_loss_analysis.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| Author(s) |
| --- |
| [Bo Zheng](https://github.com/coolalexzb) |
| [Jason Dai](https://github.com/jsondai) |
| [Naveksha Sood](https://github.com/navekshasood) |

## Overview
This notebook demonstrates how to use the Vertex Gen AI Evaluation SDK to evaluate Generative AI Agents. Whether your agent is running locally in your notebook or deployed as a managed service on Vertex AI Agent Engine, this SDK provides a unified way to simulate multi-turn conversations and assess agent performance using specialized out of the box multi-turn metrics as well as custom metrics.

We will explore two main scenarios:
<ol>
<li> <b>Local Agent Eval with User Simulation:</b> Evaluating a local ADK agent by simulating a user interacting with it over multiple turns.
<ul>
<li> <b> Predefined Metrics & Auto-Loss Analysis: </b> Evaluate the agent using out-of-the-box metrics designed for multi-turn agent evaluation, followed by a diagnostic Auto-loss Analysis to group failures into semantic loss clusters. </li>

<li> <b> Custom Registered Metrics: </b> Customize computation-based and LLM-based metrics, register them in the metric registry, and use them directly for evaluation.</li>
</ul></li>
<li> <b> Cloud Agent Eval with User Simulation:</b> Triggering an evaluation run for an agent already deployed on Vertex AI Agent Engine with predefined metrics and custom registered metrics, and take auto loss analysis in the same evaluation run  </li></ol>

## Getting Started

### Install Vertex AI SDK for Gen AI Evaluation Service
Run the following to install the necessary SDKs.


In [ ]:
%pip install --upgrade --force-reinstall -q google-cloud-aiplatform[evaluation]>=1.148.1

### Authenticate your notebook environment (Colab only)
If you're running this notebook on Google Colab, run the cell below to authenticate your environment.


In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information
Initialize the Vertex AI client.

**Important Region Note:** For local agent scraping with `gemini-3` models, you must use the `global` region. However, Vertex AI Agent Engine cannot be deployed in the global region. The helper functions below will automatically handle switching to `us-central1` when deploying your agent to Agent Engine.


In [ ]:
import os
import uuid

import vertexai
from google.adk import Agent
from vertexai import Client, types

# fmt: off
PROJECT_ID = "[your-project-id]" # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
LOCATION = "" # @param {type: "string", placeholder: "global", isTemplate: true}

# Used for deploy agent engine
AGENT_ENGINE_GCS_BUCKET="" # @param {type: "string", placeholder: "[your-gcs-bucket]", isTemplate: true}
# Used for create evaluation run
EVAL_RUN_GCS_BUCKET="" # @param {type: "string", placeholder: "[your-gcs-bucket]", isTemplate: true}
# fmt: on

client = Client(
    project=PROJECT_ID,
    location=LOCATION,
)

# Set environment variables to explicitly use Vertex AI with the ADK
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

## Helper Functions

The following cell defines two helper functions:
* `poll_evaluation_run`: A utility to monitor the status of a long-running evaluation job on Vertex AI.
* `deploy_adk_agent`: A utility to package and deploy your local agent to Vertex AI Agent Engine. Note: agent engine cannot be deployed to `global` region


In [ ]:
def poll_evaluation_run(evaluation_run, client):
    import time

    from IPython.display import clear_output

    start_time = time.time()
    current_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    terminal_states = ["SUCCEEDED", "FAILED", "CANCELLED"]

    while current_run.state not in terminal_states:
        clear_output(wait=True)
        elapsed = int(time.time() - start_time)
        print(
            f"Evaluation started at: {time.strftime('%H:%M:%S', time.localtime(start_time))}, Elapsed time: {elapsed} seconds"
        )
        current_run.show()

        time.sleep(5)
        current_run = client.evals.get_evaluation_run(name=evaluation_run.name)

    clear_output(wait=True)
    print(f"Evaluation job completed in {int(time.time() - start_time)} seconds.")
    final_result = client.evals.get_evaluation_run(
        name=evaluation_run.name, include_evaluation_items=True
    )
    final_result.show()


def deploy_adk_agent(agent, location):
    """Deploy agent to agent engine"""
    app = vertexai.agent_engines.AdkApp(
        agent=agent,
        app_name=agent.name,
    )

    ae_client = Client(project=PROJECT_ID, location=location)

    print(
        "Deploying the agent to Agent Engine. This can take up to 10 mins.", flush=True
    )
    remote_app = ae_client.agent_engines.create(
        agent=app,
        config={
            "display_name": agent.name,
            "staging_bucket": AGENT_ENGINE_GCS_BUCKET,
            "requirements": ["google-cloud-aiplatform[adk,agent_engines]"],
            "env_vars": {"GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true"},
        },
    )

    return remote_app

## Agent Preparation

Before we can evaluate, we need an agent. We will set up a travel agent built using the Google Agent Development Kit (ADK). This agent acts as a primary orchestrator that delegates tasks to two sub-agents: a `flight_agent` and a `hotel_agent`.


In [ ]:
# @title Local ADK Agent
FLIGHT_DB = [
    {
        "flight_id": "UA123",
        "airline": "United",
        "origin": "San Francisco",
        "destination": "New York",
        "date": "2025-12-20",
        "price": 500,
        "stops": 0,
        "departure": "08:00 AM",
    },
    {
        "flight_id": "BA888",
        "airline": "British Airways",
        "origin": "San Francisco",
        "destination": "London",
        "date": "2025-11-05",
        "price": 750,
        "stops": 0,
        "departure": "04:00 PM",
    },
    {
        "flight_id": "AF999",
        "airline": "Air France",
        "origin": "San Francisco",
        "destination": "Paris",
        "date": "2025-11-06",
        "price": 820,
        "stops": 1,
        "departure": "02:30 PM",
    },
    {
        "flight_id": "AA102",
        "airline": "American",
        "origin": "San Francisco",
        "destination": "Chicago",
        "date": "2025-12-01",
        "price": 280,
        "stops": 0,
        "departure": "09:15 AM",
    },
    {
        "flight_id": "DL203",
        "airline": "Delta",
        "origin": "San Francisco",
        "destination": "Miami",
        "date": "2025-12-05",
        "price": 310,
        "stops": 1,
        "departure": "11:45 AM",
    },
    {
        "flight_id": "QF304",
        "airline": "Qantas",
        "origin": "San Francisco",
        "destination": "Sydney",
        "date": "2025-12-15",
        "price": 1100,
        "stops": 0,
        "departure": "10:30 PM",
    },
    {
        "flight_id": "NH405",
        "airline": "ANA",
        "origin": "San Francisco",
        "destination": "Tokyo",
        "date": "2025-11-25",
        "price": 950,
        "stops": 0,
        "departure": "01:15 PM",
    },
    {
        "flight_id": "UA506",
        "airline": "United",
        "origin": "San Francisco",
        "destination": "Berlin",
        "date": "2025-12-10",
        "price": 890,
        "stops": 1,
        "departure": "05:00 PM",
    },
    {
        "flight_id": "B6607",
        "airline": "JetBlue",
        "origin": "San Francisco",
        "destination": "Boston",
        "date": "2025-11-12",
        "price": 290,
        "stops": 0,
        "departure": "07:00 AM",
    },
    {
        "flight_id": "AS708",
        "airline": "Alaska Airlines",
        "origin": "San Francisco",
        "destination": "Seattle",
        "date": "2025-11-18",
        "price": 150,
        "stops": 0,
        "departure": "08:30 AM",
    },
    {
        "flight_id": "JL666",
        "airline": "Japan Airlines",
        "origin": "San Francisco",
        "destination": "Tokyo",
        "date": "2025-11-10",
        "price": 1200,
        "stops": 0,
        "departure": "01:00 PM",
    },
]

HOTEL_DB = [
    {
        "id": "HT1",
        "name": "The Grand Hotel",
        "city": "Chicago",
        "rating": 4.5,
        "amenities": ["Pool", "Free WiFi", "Gym"],
        "room_types": {"Standard": 150, "Suite": 300},
        "distance_to_center": "0.5 miles",
    },
    {
        "id": "HT2",
        "name": "Budget Inn",
        "city": "Chicago",
        "rating": 3.0,
        "amenities": ["Free Breakfast"],
        "room_types": {"Standard": 80},
        "distance_to_center": "5.0 miles",
    },
    {
        "id": "HT3",
        "name": "Parisian Palace",
        "city": "Paris",
        "rating": 5.0,
        "amenities": ["Spa", "Eiffel Tower View", "Fine Dining"],
        "room_types": {"Deluxe": 240, "Penthouse": 800},
        "distance_to_center": "1.2 miles",
    },
    {
        "id": "HT4",
        "name": "Montmartre Hostel",
        "city": "Paris",
        "rating": 4.1,
        "amenities": ["Free WiFi", "Walking Tours"],
        "room_types": {"Bunk": 40, "Private": 110},
        "distance_to_center": "2.0 miles",
    },
    {
        "id": "HT5",
        "name": "Golden Gate Inn",
        "city": "San Francisco",
        "rating": 4.2,
        "amenities": ["Free WiFi", "Gym"],
        "room_types": {"Standard": 200, "Suite": 400},
        "distance_to_center": "2.0 miles",
    },
    {
        "id": "HT6",
        "name": "Bay View Suites",
        "city": "San Francisco",
        "rating": 4.8,
        "amenities": ["Ocean View", "Spa", "Valet"],
        "room_types": {"Deluxe": 350},
        "distance_to_center": "0.3 miles",
    },
    {
        "id": "HT7",
        "name": "Times Square Hotel",
        "city": "New York",
        "rating": 4.6,
        "amenities": ["City View", "Gym", "Bar"],
        "room_types": {"Standard": 280, "Deluxe": 450},
        "distance_to_center": "0.1 miles",
    },
    {
        "id": "HT8",
        "name": "Brooklyn Backpacker",
        "city": "New York",
        "rating": 3.8,
        "amenities": ["Free Breakfast", "Laundry"],
        "room_types": {"Standard": 110},
        "distance_to_center": "4.5 miles",
    },
    {
        "id": "HT9",
        "name": "Hollywood Blvd Resort",
        "city": "Los Angeles",
        "rating": 4.3,
        "amenities": ["Pool", "Valet Parking"],
        "room_types": {"Standard": 220, "Suite": 500},
        "distance_to_center": "3.5 miles",
    },
    {
        "id": "HT10",
        "name": "Berlin Central",
        "city": "Berlin",
        "rating": 4.1,
        "amenities": ["Free WiFi", "Bar", "Bike Rental"],
        "room_types": {"Standard": 130},
        "distance_to_center": "0.8 miles",
    },
    {
        "id": "HT11",
        "name": "Colosseum Views",
        "city": "Rome",
        "rating": 4.7,
        "amenities": ["Historic Building", "Free Breakfast"],
        "room_types": {"Standard": 180, "Deluxe": 260},
        "distance_to_center": "1.0 miles",
    },
    {
        "id": "HT12",
        "name": "Ramblas Boutique",
        "city": "Barcelona",
        "rating": 4.4,
        "amenities": ["Rooftop Terrace", "Pool"],
        "room_types": {"Standard": 160, "Suite": 320},
        "distance_to_center": "0.2 miles",
    },
    {
        "id": "HT13",
        "name": "Thames Riverside",
        "city": "London",
        "rating": 4.6,
        "amenities": ["River View", "Gym", "Spa"],
        "room_types": {"Standard": 250, "Deluxe": 400},
        "distance_to_center": "1.5 miles",
    },
    {
        "id": "HT14",
        "name": "Shinjuku Capsule",
        "city": "Tokyo",
        "rating": 4.0,
        "amenities": ["Public Bath", "Free WiFi"],
        "room_types": {"Capsule": 60},
        "distance_to_center": "0.5 miles",
    },
    {
        "id": "HT15",
        "name": "Tokyo Tower Luxury",
        "city": "Tokyo",
        "rating": 4.9,
        "amenities": ["City View", "Fine Dining", "Spa"],
        "room_types": {"Deluxe": 400, "Suite": 1200},
        "distance_to_center": "2.1 miles",
    },
    {
        "id": "HT16",
        "name": "South Beach Resort",
        "city": "Miami",
        "rating": 4.6,
        "amenities": ["Beachfront", "Pool", "Spa"],
        "room_types": {"Standard": 250, "Ocean View": 450},
        "distance_to_center": "4.0 miles",
    },
    {
        "id": "HT17",
        "name": "Miami Downtown Inn",
        "city": "Miami",
        "rating": 3.9,
        "amenities": ["Free WiFi", "Gym"],
        "room_types": {"Standard": 120},
        "distance_to_center": "0.5 miles",
    },
    {
        "id": "HT18",
        "name": "Sydney Harbour Hotel",
        "city": "Sydney",
        "rating": 4.8,
        "amenities": ["Harbour View", "Fine Dining", "Pool"],
        "room_types": {"Standard": 300, "Suite": 600},
        "distance_to_center": "1.0 miles",
    },
    {
        "id": "HT19",
        "name": "Bondi Beach Backpackers",
        "city": "Sydney",
        "rating": 4.2,
        "amenities": ["Free Breakfast", "Surfboard Rental"],
        "room_types": {"Bunk": 50, "Private": 120},
        "distance_to_center": "4.5 miles",
    },
    {
        "id": "HT20",
        "name": "Historic Boston Suites",
        "city": "Boston",
        "rating": 4.5,
        "amenities": ["Free WiFi", "Valet Parking", "Restaurant"],
        "room_types": {"Standard": 220, "Suite": 380},
        "distance_to_center": "0.2 miles",
    },
    {
        "id": "HT21",
        "name": "Boston Harbor Inn",
        "city": "Boston",
        "rating": 4.1,
        "amenities": ["Gym", "Business Center"],
        "room_types": {"Standard": 180},
        "distance_to_center": "1.5 miles",
    },
    {
        "id": "HT22",
        "name": "Space Needle View Hotel",
        "city": "Seattle",
        "rating": 4.7,
        "amenities": ["City View", "Free WiFi", "Gym"],
        "room_types": {"Standard": 210, "Suite": 400},
        "distance_to_center": "0.8 miles",
    },
    {
        "id": "HT23",
        "name": "Pike Place Lodge",
        "city": "Seattle",
        "rating": 4.0,
        "amenities": ["Free Breakfast", "Laundry"],
        "room_types": {"Standard": 140},
        "distance_to_center": "0.5 miles",
    },
]


USER_BOOKINGS = []


def search_flights(origin: str, destination: str):
    """Searches for flights and returns a high-level summary of available options.

    Args:
        origin: The origin city name of the flight, e.g., San Francisco.
        destination: The destination city name of the flight, e.g., New York.

    Returns:
        A list of available flights.
    """
    results = []
    for f in FLIGHT_DB:
        if (
            f["origin"].lower() == origin.lower()
            and f["destination"].lower() == destination.lower()
        ):
            results.append(
                {
                    "flight_id": f["flight_id"],
                    "airline": f["airline"],
                    "starting_price": f"${f['price']}",
                }
            )
    return results


def get_flight_details(flight_id: str):
    """Gets the full details, baggage policies, and class options for a specific flight."""
    for f in FLIGHT_DB:
        if f["flight_id"] == flight_id:
            return {"details": f}
    return {"error": "Flight not found."}


def book_flight(flight_id: str, seat_class: str):
    """Books a specific flight for a passenger in the chosen seat class.

    Args:
        flight_id: The unique identifier for the flight.
        seat_class: The chosen seat class (e.g., 'Economy', 'Business', 'First Class').

    Returns:
        A dictionary containing the booking status, booking ID, and total bookings, or an error message.
    """
    for f in FLIGHT_DB:
        if f["flight_id"] == flight_id:
            # Provide a default list if "class_options" is missing
            available_classes = f.get(
                "class_options",
                [
                    "economy",
                    "business",
                    "first class",
                    "Economy",
                    "Business",
                    "First Class",
                ],
            )
            if seat_class not in available_classes:
                return {
                    "error": f"Seat class '{seat_class}' not available. Options: {available_classes}"
                }

            booking_id = f"BKG-F-{uuid.uuid4().hex[:6]}"
            USER_BOOKINGS.append(
                {
                    "type": "flight",
                    "booking_id": booking_id,
                    "flight_id": flight_id,
                    "class": seat_class,
                }
            )
            return {
                "status": "Success",
                "booking_id": booking_id,
                "total_bookings": len(USER_BOOKINGS),
            }
    return {"error": "Flight ID not found."}


# --- HOTEL FUNCTIONS ---
def search_hotels(city: str, min_rating: float = 0.0):
    """Searches for hotels in a specific city, optionally filtering by minimum rating.

    Args:
        city: The name of the city to search for hotels.
        min_rating: The minimum acceptable rating for the hotel. Defaults to 0.0.

    Returns:
        A dictionary containing a list of hotels matching the search criteria.
    """
    results = []
    for h in HOTEL_DB:
        if h["city"].lower() == city.lower() and h["rating"] >= min_rating:
            results.append({"id": h["id"], "name": h["name"], "rating": h["rating"]})
    return {"hotels": results}


def get_hotel_details(hotel_id: str):
    """Gets detailed information for a hotel, including room types, pricing, and amenities.

    Args:
        hotel_id: The unique identifier for the hotel.

    Returns:
        A dictionary containing the detailed hotel information, or an error message if not found.
    """
    for h in HOTEL_DB:
        if h["id"] == hotel_id:
            return {"details": h}
    return {"error": "Hotel not found."}


def book_hotel(hotel_id: str, room_type: str):
    """Books a specific room type at a hotel for the given dates.

    Args:
        hotel_id: The unique identifier for the hotel.
        room_type: The type of room to book (e.g., 'Standard', 'Deluxe').

    Returns:
        A dictionary containing the booking status, booking ID, and total bookings, or an error message.
    """
    for h in HOTEL_DB:
        if h["id"] == hotel_id:
            if room_type not in h["room_types"]:
                return {
                    "error": f"Room type '{room_type}' not available. Options: {list(h['room_types'].keys())}"
                }
            booking_id = f"BKG-H-{uuid.uuid4().hex[:6]}"
            USER_BOOKINGS.append(
                {
                    "type": "hotel",
                    "booking_id": booking_id,
                    "hotel_id": hotel_id,
                    "room": room_type,
                }
            )
            return {
                "status": "Success",
                "booking_id": booking_id,
                "total_bookings": len(USER_BOOKINGS),
            }
    return {"error": "Hotel ID not found."}


# 1. The Domain Specialist for Flights
flight_agent = Agent(
    model="gemini-3-flash-preview",
    name="flight_specialist",
    instruction="You are a flight booking specialist. Use search_flights to find IDs, then get_flight_details to verify luggage/refund policies, and book_flight when the user confirms.",
    tools=[search_flights, get_flight_details, book_flight],
)

# 2. The Domain Specialist for Hotels
hotel_agent = Agent(
    model="gemini-3-flash-preview",
    name="hotel_specialist",
    instruction="You are a hotel booking specialist. Use search_hotels to find IDs, then get_hotel_details to compare room types and amenities, and finally book_hotel.",
    tools=[search_hotels, get_hotel_details, book_hotel],
)

# 3. The Root Orchestrator
travel_agent = Agent(
    model="gemini-3-flash-preview",  # Use a stronger reasoning model for orchestration
    name="travel_agent",
    instruction="You are a primary travel concierge. Talk to the user to understand their full itinerary. Delegate flight-related tasks to the flight_specialist, and hotel-related tasks to the hotel_specialist. Synthesize their results to the user.",
    sub_agents=[flight_agent, hotel_agent],
)

### [Optional] Deployed ADK Agent on Agent Engine
If you want to evaluate a deployed agent (Cloud Agent Eval with User Simulation), you can use the helper function to deploy your local agent to Vertex AI. This step packages your code and requirements and creates an endpoint.


In [ ]:
# Optional, deploy the agent if the agent has not been deployed to agent engine
agent_engine = deploy_adk_agent(travel_agent, "us-central1")

# 1. Local Agent Eval with User Simulation

In this scenario, we evaluate the agent running locally in our notebook memory using:
1. **Predefined Metrics & Auto-Loss Analysis**: Out-of-the box predefined metrics specially designed for multi-turn agentic evaluation, followed by loss analysis
2. **Custom Registered Metrics**: Custom metrics registered in metric registry

### Step 1: Generate synthetic conversation scenarios  

First, we use the `generate_conversation_scenarios` method to automatically create testing scenarios based on our agent's capabilities and specific instructions (e.g., a user changing their mind about a destination).

**Note**:
1. If the `model_name` is not specified, `generate_conversation_scenarios` use `gemini-3.1-pro-preview` by default. This default model prioritizes higher quality but may take several minutes to process.
2. To reduce latency with only a slight trade-off in quality, you can explicitly set the `model_name` to `gemini-3-flash-preview` Or you can test with simpler `generation_instruction` with value `Generate scenarios where the user tries to book a flight.` to make sure the generated `converstaion_plan` is less complex


In [ ]:
agent_info = types.evals.AgentInfo.load_from_agent(agent=travel_agent)

eval_dataset = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 5,
        "generation_instruction": "Generate scenarios where the user tries to book a flight but changes their mind about the destination.",
        "environment_context": "Today is Monday. I am located in San Francisco. Flights to Paris, New York, Tokyo, Chicago, Sydney, etc are available.",
    },
)
display(eval_dataset.eval_dataset_df.head())

### Step 2: Run agent locally to simulate multi-turn conversations

Next, we use a User Simulator to interact with our local agent based on the scenarios generated above.

**Note:**
1. By default, the ADK library will automatically use `gemini-2.5-flash` for the user simulator.
2. The default `max_turn` value is 5, you can increase the `max_turn` value to get full agent trace if receiving hitting limit warning.

In [ ]:
eval_dataset_with_trace = client.evals.run_inference(
    agent=travel_agent,
    src=eval_dataset,
    config={
        "user_simulator_config": {
            "max_turn": 3,
        }
    },
)
display(eval_dataset_with_trace.eval_dataset_df.head())

### Step 3: Evaluation

#### 3.1 Create and register custom metrics

##### 3.1.1 Create custom computation based metric to evaluate agent efficiency

To evaluate complex agent behaviors, you can define a custom computation-based metric using a Python function.

In the code below, we create a metric that evaluates the efficiency of our agent's multi-turn trajectory. The custom Python function parses the `agent_eval_data` to analyze the sequence of events and tool calls across all conversation turns.

Specifically, this efficiency metric calculates a score from 0.0 to 1.0 based on the following logic:
* It starts with a base score of 1.0.
* It applies a small penalty for every tool call made by the agent (encouraging the agent to reach the goal with fewer steps).
* It applies a heavier penalty for redundant or looping tool calls (e.g., calling the same function with the exact same arguments multiple times).

Once the logic is defined in a Python string snippet, it is packaged into a metric object so that it can be passed to the metric registry service.

In [ ]:
efficiency_metric_code = """
import json

def evaluate(instance: dict) -> float:
    agent_data = instance.get('agent_eval_data', {})
    turns = agent_data.get('turns', [])

    total_calls = 0
    seen_calls = set()
    redundant_calls = 0

    for turn in turns:
        for event in turn.get('events', []):
            content = event.get('content', event)
            if content.get('role') == 'model':
                for part in content.get('parts', []):
                    fc = part.get('function_call')
                    if fc:
                        total_calls += 1
                        # Create a unique key for the call (name + args) to detect loops
                        call_key = (fc.get('name'), json.dumps(fc.get('args'), sort_keys=True))
                        if call_key in seen_calls:
                            redundant_calls += 1
                        seen_calls.add(call_key)

    # Demo Scoring Logic:
    # 1. Base Score: 1.0
    # 2. Penalty: -0.02 for every internal function call
    # 3. Penalty: -0.10 for every redundant call (looping behavior)

    score = 1.0 - (total_calls * 0.02) - (redundant_calls * 0.10)

    # Ensure score stays between 0 and 1
    return max(0.0, min(1.0, score))
"""

efficiency_metric = types.CodeExecutionMetric(
    name="multi_turn_efficiency", custom_function=efficiency_metric_code
)

Next, you can register the custom metric by calling `create_evaluation_metric()`, the metric definition is saved in the cloud, and the service returns a unique resource name. You could then reference this resource name in any future evaluation runs.

In [ ]:
efficiency_metric_path = client.evals.create_evaluation_metric(metric=efficiency_metric)
print("Metric has been registered at: ", efficiency_metric_path)

registered_efficiency_metric = types.Metric(
    name="multi_turn_efficiency", metric_resource_name=efficiency_metric_path
)

##### 3.1.2 Create custom LLM based metric for checking the tone of agent responses

In addition to computation-based metrics, you can also define custom LLM-based metrics using `LLMMetric` object. This allows you to leverage an LLM "judge" to evaluate qualitative aspects of an agent's performance, such as tone or helpfulness.

In the example below, we author a metric to evaluate the agent's tone. The `LLMMetric` is configured with:
* **Instructions and Criteria**: A prompt template that defines specific dimensions—Professionalism and Empathy—for the judge to evaluate.
* **Input Variables**: Placeholders like `{agent_data}` that the service populates with the actual conversation trace during evaluation
* **Structured Output Format**: Instructions for the LLM judge to return a JSON list of objects containing verdicts and reasoning for each property.
* **Custom Result Parsing**: A Python function that extracts the judge's JSON output, calculates a final numerical score based on the verdicts, and consolidates the reasoning into a single explanation.


In [ ]:
tone_check_metric = types.LLMMetric(
    name="tone_check",
    prompt_template="""Analyze the tone of the response based on these two criteria:\n
          1. Professionalism: The response should use appropriate language and maintain a business-like demeanor.\n
          2. Empathy: The response should acknowledge the user's feelings and show understanding.\n\n
          Prompt: {agent_data.turns[0].events[0]}
          Response: {agent_data.turns[0].events[1]}
          Return ONLY a JSON list of objects for these two properties:
          [{"property": "Professionalism", "verdict": true, "reasoning": "..."},
          {"property": "Empathy", "verdict": true, "reasoning": "..."}]
        """,
    result_parsing_function="""
          import json, re
          def parse_results(responses):
              text = responses[0]
              # Use robust regex to find the JSON list block
              match = re.search("[\\[].*[]]", text, re.DOTALL)
              if not match: return {"score": 0.0, "explanation": "No valid JSON found"}

              try:
                  data = json.loads(match.group(0))
                  # Calculate an overall score (e.g., average of verdicts)
                  passed_count = sum(1 for r in data if r.get("verdict", False))
                  total_count = len(data)
                  score = passed_count / total_count if total_count > 0 else 0.0

                  # Consolidate reasoning into a single explanation string
                  explanation = "\\n".join([f"{r.get('property')}: {r.get('reasoning')}" for r in data])

                  # IMPORTANT: Return a dictionary, not a list
                  return {
                      "score": float(score),
                      "explanation": explanation
                  }
              except Exception as e:
                  return {"score": 0.0, "explanation": f"Parsing failed: {str(e)}"}
          """,
)

Register the custom LLM-based metric.

In [ ]:
tone_check_metric_path = client.evals.create_evaluation_metric(metric=tone_check_metric)
print("Metric has been registered at: ", tone_check_metric_path)

registered_tone_check_metric = types.Metric(
    name="tone-check", metric_resource_name=tone_check_metric_path
)

#### 3.2 Evaluate Predefined Metrics and Custom Metrics

You can use these custom defined metrics as well as predefined multi-turn metrics to measure tool use quality, trajectory quality, and overall task success and some custom criteria to evaluation the agent performance.

**Note**: single-turn metrics do not work with simulated multi-turn agent_data. If you wish to use single-turn metrics like `GENERAL_QUALITY` or `SAFETY`, please keep `max_turn` at 1 in `user_simulator_config` in the previous step.


In [ ]:
eval_metrics = [
    registered_efficiency_metric,
    registered_tone_check_metric,
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TRAJECTORY_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

eval_result = client.evals.evaluate(
    dataset=eval_dataset_with_trace,
    metrics=eval_metrics,
)

In [ ]:
eval_result.show()

## Step 4: Generate loss Clusters

Finally, Generate insights from  evaluation results using `generate_loss_clusters` method. This function analyzes the traces that failed and groups them into semantic loss clusters to help you identify common failure patterns.

**Note:**
1. Currently, the Auto-loss Analysis recipe is only supported for two specific predefined metrics: `MULTI_TURN_TASK_SUCCESS` and `MULTI_TURN_TOOL_USE_QUALITY`. Passing other metrics into this function may not yield classification results.
2. "Other NA" on dashboard result means "The evaluation result cannot be categorized to a loss cluster"

In [ ]:
loss_analysis = client.evals.generate_loss_clusters(
    eval_result=eval_result,
    metric=types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
)

print(f"Loss analysis completed! {len(loss_analysis.results or [])} result(s).")
loss_analysis.show()

#  2. Cloud Agent Eval with User Simulation

In this scenario, we evaluate the agent that we previously deployed to Vertex AI Agent Engine. This simulates how you would evaluate a production-ready application.


##  Step 1: Generate synthetic user scenarios

In [ ]:
agent_info = types.evals.AgentInfo.load_from_agent(agent=travel_agent)

eval_dataset = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 5,
        "generation_instruction": "Generate scenarios where the user tries to book a flight but changes their mind about the destination.",
        "environment_context": "Today is Monday. I am located in San Francisco. Flights to Paris, New York, Tokyo, Chicago, Sydney, etc are available.",
    },
)
display(eval_dataset.eval_dataset_df.head())

## Step 2: Trigger evaluation run from eval management service

We trigger a remote evaluation job using the Eval Management Service. This evaluation run orchestrates a comprehensive, workflow:

1. **Agent Execution**: Automatically kicks off the agent deployed on Agent Engine to interact with the simulated user and collect conversation traces. You can fill `agent` parameter with existing agent resource name if deploy agent engine step is skipped, for example, `projects/{project_number}/locations/{location}/reasoningEngines/{agent_engine_id}`
2. **Agent Evaluation**: Evaluate both predefined agent eval metrics and custom registered metrics
3. **Loss Analysis**: Perform loss analysis in the evaluation run via specifying `loss_analysis_metrics`

**Note:**
1. The user simulator will default to using `gemini-2.5-flash`, so `model_name` is omitted from `create_evaluation_run` and the default `max_turn` value is 5
3. The evaluation job might take a long time (several minutes or more) as we bundle all new features in the job. You could test with less eval metrics, less complex agent, less complex `generation_instruction` (in the previous step) or remove loss analysis step to get eval results faster.


In [ ]:
eval_metrics = [
    registered_efficiency_metric,
    registered_tone_check_metric,
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TRAJECTORY_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

loss_analysis_metrics = [
    types.RubricMetric.MULTI_TURN_TOOL_USE_QUALITY,
    types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
]

# Eval Run will run agent, generate traces, and then calculate the metrics.
eval_run = client.evals.create_evaluation_run(
    agent=agent_engine.api_resource.name,
    agent_info=agent_info,
    dataset=eval_dataset,
    metrics=eval_metrics,
    dest=EVAL_RUN_GCS_BUCKET,
    user_simulator_config={
        "max_turn": 3,
    },
    loss_analysis_metrics=loss_analysis_metrics,
)

In [ ]:
poll_evaluation_run(eval_run, client)